# Face Mask Classification — MobileNetV2 (Colab)

Trains a **MobileNetV2** classifier on cropped faces from the [MAKS Kaggle dataset](https://www.kaggle.com/datasets/ashishjangra27/face-mask-12k-images-dataset).

**Classes:** `with_mask`, `mask_weared_incorrect`, `without_mask`

**Strategy:** ImageNet-pretrained backbone is fully frozen; only the classification head is trained for fast Colab GPU runs.

### Before you run
1. **Runtime → Change runtime type → T4 GPU**
2. Upload the whole `cv_final_project` folder to Google Drive (you already did this)
3. In section 1 below, set `PROJECT_DIR` to your Drive path, then run all cells
4. After training, download `face_mask_artifacts.zip` (or copy `artifacts/` from Drive) back to your Mac

In [ ]:
!pip -q install tensorflow scikit-learn matplotlib Pillow

In [ ]:
import json
import random
import xml.etree.ElementTree as ET
import zipfile
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from google.colab import files
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Mount Google Drive & locate project

Your whole `cv_final_project` folder is on Drive. Mount Drive, set `PROJECT_DIR`, and the notebook will use `archive-4/` from there (no zip upload needed).

In [ ]:
from google.colab import drive

# --- EDIT THIS if your folder is not in My Drive root ---
PROJECT_DIR = Path('/content/drive/MyDrive/cv_final_project')

drive.mount('/content/drive')

if not PROJECT_DIR.exists():
    print('PROJECT_DIR not found. Searching Drive for cv_final_project...')
    matches = [
        p for p in Path('/content/drive/MyDrive').rglob('cv_final_project')
        if (p / 'archive-4' / 'images').exists()
    ]
    for p in matches:
        print(' ', p)
    if len(matches) == 1:
        PROJECT_DIR = matches[0]
        print('Auto-selected:', PROJECT_DIR)
    else:
        raise FileNotFoundError(f'Update PROJECT_DIR — not found: {PROJECT_DIR}')

DATA_DIR = PROJECT_DIR / 'archive-4'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

assert (DATA_DIR / 'images').exists(), f'Missing images/ in {DATA_DIR}'
assert (DATA_DIR / 'annotations').exists(), f'Missing annotations/ in {DATA_DIR}'
print('Project root:', PROJECT_DIR.resolve())
print('Dataset root:', DATA_DIR.resolve())
print('Artifacts will save to:', ARTIFACT_DIR.resolve())

## 2. Configuration

In [ ]:
CLASS_NAMES = ('with_mask', 'mask_weared_incorrect', 'without_mask')
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
IMG_SIZE = (224, 224)

BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-3
VAL_RATIO = 0.2
SEED = 42
UNFREEZE_LAST_N_LAYERS = 0  # keep 0 for fastest training; try 10-20 for a small accuracy boost

# ARTIFACT_DIR is set in the Drive mount cell above

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Parse annotations → face crops

In [ ]:
@dataclass(frozen=True)
class FaceSample:
    image_path: Path
    label: str
    bbox: tuple[int, int, int, int]


def parse_annotations(data_dir: Path) -> list[FaceSample]:
    samples = []
    for xml_path in sorted((data_dir / 'annotations').glob('*.xml')):
        root = ET.parse(xml_path).getroot()
        filename = root.findtext('filename')
        image_path = data_dir / 'images' / filename
        if not image_path.exists():
            continue
        for obj in root.findall('object'):
            label = obj.findtext('name')
            if label not in CLASS_TO_INDEX:
                continue
            box = obj.find('bndbox')
            xmin = int(float(box.findtext('xmin')))
            ymin = int(float(box.findtext('ymin')))
            xmax = int(float(box.findtext('xmax')))
            ymax = int(float(box.findtext('ymax')))
            if xmax > xmin and ymax > ymin:
                samples.append(FaceSample(image_path, label, (xmin, ymin, xmax, ymax)))
    return samples


def stratified_split(samples, val_ratio=0.2, seed=42):
    rng = random.Random(seed)
    by_class = {name: [] for name in CLASS_NAMES}
    for s in samples:
        by_class[s.label].append(s)
    train, val = [], []
    for class_samples in by_class.values():
        rng.shuffle(class_samples)
        n_val = max(1, int(len(class_samples) * val_ratio))
        val.extend(class_samples[:n_val])
        train.extend(class_samples[n_val:])
    rng.shuffle(train)
    rng.shuffle(val)
    return train, val


def crop_face(sample: FaceSample, margin=0.15) -> np.ndarray:
    image = Image.open(sample.image_path).convert('RGB')
    xmin, ymin, xmax, ymax = sample.bbox
    w, h = image.size
    pad_x = int((xmax - xmin) * margin)
    pad_y = int((ymax - ymin) * margin)
    xmin, ymin = max(0, xmin - pad_x), max(0, ymin - pad_y)
    xmax, ymax = min(w, xmax + pad_x), min(h, ymax + pad_y)
    crop = image.crop((xmin, ymin, xmax, ymax)).resize(IMG_SIZE, Image.BILINEAR)
    return np.asarray(crop, dtype=np.uint8)


samples = parse_annotations(DATA_DIR)
train_samples, val_samples = stratified_split(samples, VAL_RATIO, SEED)

from collections import Counter
print(f'Total face crops: {len(samples)}')
print('Train:', Counter(s.label for s in train_samples))
print('Val:  ', Counter(s.label for s in val_samples))

## 4. tf.data pipeline

In [ ]:
def make_dataset(face_samples, batch_size, shuffle=False, augment=False):
    records = [
        {
            'image_path': str(s.image_path),
            'label': CLASS_TO_INDEX[s.label],
            'bbox': s.bbox,
        }
        for s in face_samples
    ]

    def gen():
        for r in records:
            yield r

    sig = {
        'image_path': tf.TensorSpec((), tf.string),
        'label': tf.TensorSpec((), tf.int32),
        'bbox': tf.TensorSpec((4,), tf.int32),
    }
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    if shuffle:
        ds = ds.shuffle(min(len(records), 1024))

    def load_crop(rec):
        def _py(path, box):
            if hasattr(path, 'numpy'):
                path = path.numpy()
            if isinstance(path, bytes):
                path = path.decode('utf-8')
            else:
                path = str(path)
            if hasattr(box, 'numpy'):
                box = box.numpy()
            s = FaceSample(path, 'with_mask', tuple(int(v) for v in box))
            return crop_face(s)
        img = tf.py_function(_py, [rec['image_path'], rec['bbox']], tf.uint8)
        img.set_shape((*IMG_SIZE, 3))
        return tf.cast(img, tf.float32), rec['label']

    def augment_fn(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.15)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        return tf.clip_by_value(img, 0.0, 255.0), label

    def preprocess(img, label):
        return tf.keras.applications.mobilenet_v2.preprocess_input(img), label

    ds = ds.map(load_crop, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(train_samples, BATCH_SIZE, shuffle=True, augment=True)
val_ds = make_dataset(val_samples, BATCH_SIZE, shuffle=False, augment=False)
print('Train batches:', len(train_samples) // BATCH_SIZE)
print('Val batches:  ', len(val_samples) // BATCH_SIZE)

## 5. Build MobileNetV2 (frozen backbone)

In [ ]:
inputs = layers.Input(shape=(*IMG_SIZE, 3))
base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights='imagenet',
    input_tensor=inputs,
)
base_model.trainable = False

if UNFREEZE_LAST_N_LAYERS > 0:
    for layer in base_model.layers[-UNFREEZE_LAST_N_LAYERS:]:
        layer.trainable = True

x = layers.GlobalAveragePooling2D(name='gap')(base_model.output)
x = layers.Dropout(0.3, name='dropout')(x)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax', name='predictions')(x)
model = models.Model(inputs, outputs, name='face_mask_mobilenet')

model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

trainable = sum(int(l.trainable) for l in model.layers)
total_params = model.count_params()
trainable_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Trainable top-level layers: {trainable}/{len(model.layers)}')
print(f'Trainable params: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)')
model.summary()

## 6. Train

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
    ModelCheckpoint(
        ARTIFACT_DIR / 'best_face_mask_mobilenet.keras',
        monitor='val_accuracy',
        save_best_only=True,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss'); plt.legend()
plt.tight_layout()
plt.show()

## 7. Evaluate

In [ ]:
best_model = tf.keras.models.load_model(ARTIFACT_DIR / 'best_face_mask_mobilenet.keras')

y_true, y_pred = [], []
for images, labels in val_ds:
    probs = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

print(classification_report(y_true, y_pred, target_names=list(CLASS_NAMES)))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right')
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout()
plt.show()

## 8. Save & download artifacts
Download the zip and place it in your local project under `artifacts/`.

In [ ]:
best_model.save(ARTIFACT_DIR / 'face_mask_mobilenet.keras')
best_model.save_weights(ARTIFACT_DIR / 'face_mask_mobilenet.weights.h5')

(ARTIFACT_DIR / 'class_names.json').write_text(json.dumps(list(CLASS_NAMES), indent=2))
(ARTIFACT_DIR / 'history.json').write_text(
    json.dumps({k: [float(v) for v in vals] for k, vals in history.history.items()}, indent=2)
)
(ARTIFACT_DIR / 'config.json').write_text(json.dumps({
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'unfreeze_last_n_layers': UNFREEZE_LAST_N_LAYERS,
    'class_names': list(CLASS_NAMES),
}, indent=2))

print('Saved to Google Drive:', ARTIFACT_DIR)
for f in sorted(ARTIFACT_DIR.iterdir()):
    print(' ', f.name)

# Optional: also download a zip to your Mac
zip_path = '/content/face_mask_artifacts.zip'
!cd "{ARTIFACT_DIR}" && zip -r "{zip_path}" .
files.download(zip_path)
print('Also downloaded zip to your Mac')